# 🤖 AI Engineering Fundamentals — Lezione 3
## Notebook Gruppo C

**ITS Novitas 4.0 | Martedì 26/05/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [1]:
GRUPPO = "C"
MEMBRI = ["Jacopo Zucca", "Giulio Pazzola", "Guido Righi"]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

Gruppo C — Jacopo Zucca, Giulio Pazzola, Guido Righi


In [2]:
!pip install anthropic -q
from google.colab import userdata
import anthropic, os, json

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
print("✅ Setup completato!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 833.0/833.0 kB 12.4 MB/s eta 0:00:00
✅ Setup completato!


---
## 🎯 Tema del Gruppo C: Memoria Persistente & Streaming

Costruite un chatbot che ricorda tra sessioni diverse
e risponde in tempo reale con lo streaming.

---
### Esercizio 1 — Memoria persistente su JSON *(guidato)*

Salvate la history su file JSON e ricaricatela.
Simulate due sessioni separate — il chatbot deve ricordare
quello che avete detto nella sessione precedente.

In [ ]:
# Esercizio 1 — memoria persistente

MEMORY_FILE = "chat_widata.json"

def carica_storia():
    """Carica la history dal file JSON. Restituisce lista vuota se non esiste."""
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            storia = json.load(f)
            print(f"📂 Storia caricata: {len(storia)} messaggi precedenti")
            return storia
    print("🆕 Nessuna storia precedente — nuova conversazione")
    return []

def salva_storia(history):
    """Salva la history su file JSON."""
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    print(f"💾 Storia salvata: {len(history)} messaggi")

def chat_persistente(messaggio, history):
    history.append({"role": "user", "content": messaggio})
    risposta = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=history
    )
    testo = risposta.content[0].text
    history.append({"role": "assistant", "content": testo})
    salva_storia(history)  # salva dopo ogni messaggio
    return testo

# ── SESSIONE 1 ────────────────────────────────────────────────────
print("=" * 50)
print("SESSIONE 1")
print("=" * 50)

history = carica_storia()
print(f"Messaggi precedenti caricati: {len(history)}")

r = chat_persistente("Ciao! Mi chiamo Jacopo e sto imparando AI Engineering.", history)
print(f"🤖 {r}\n")
r = chat_persistente("Il mio progetto finale è un chatbot per WiData Srl.", history)
print(f"🤖 {r}\n")

print(f"History salvata: {len(history)} messaggi\n")

# ── SESSIONE 2 — nuova sessione, stessa memoria ───────────────────
print("=" * 50)
print("SESSIONE 2 (nuova sessione)")
print("=" * 50)

history = carica_storia()  # ricarica
print(f"Messaggi precedenti caricati: {len(history)}")

r = chat_persistente("Come mi chiamo e su cosa sto lavorando?", history)
print(f"🤖 {r}")

# Il chatbot ricorda? Se sì, perché? Se no, cosa manca?
# ...

SESSIONE 1
🆕 Nessuna storia precedente — nuova conversazione
Messaggi precedenti caricati: 0
💾 Storia salvata: 2 messaggi
🤖 Ciao Jacopo! 👋 Bellissimo! L'AI Engineering è un campo affascinante e in crescita esponenziale.

Sono qui per aiutarti. Posso supportarti con:

- **Concetti fondamentali**: machine learning, deep learning, NLP, computer vision
- **Programmazione**: Python, framework come TensorFlow, PyTorch, scikit-learn
- **Prompt engineering e LLMs**: come lavorare con modelli come me
- **Architetture**: reti neurali, transformer, retrieval-augmented generation (RAG)
- **Best practices**: data preprocessing, model evaluation, deployment
- **Progettetti e casi d'uso**: dalla teoria alla pratica

**Qualche domanda per iniziare:**
- A che livello sei (principiante, intermedio, avanzato)?
- Su quale area ti concentri maggiormente?
- Hai un progetto specifico in mente o stai ancora esplorando?

Fammi sapere come posso essere più utile! 🚀

💾 Storia salvata: 4 messaggi
🤖 Fantastico! 🎯 

---
### Esercizio 2 — Streaming *(guidato)*

Implementate lo streaming e confrontate la differenza percepita
rispetto alla chiamata normale. Testate con e senza `flush=True`.

In [ ]:
# Esercizio 2 — streaming
import time

domanda_lunga = "Spiegami in dettaglio come funziona un sistema IoT completo, dalla raccolta dei dati dei sensori fino alla visualizzazione sulla piattaforma cloud."

# ── SENZA streaming ───────────────────────────────────────────────
print("SENZA streaming:")
t_start = time.time()
risposta = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    messages=[{"role": "user", "content": domanda_lunga}]
)
t_fine = time.time()
print(risposta.content[0].text[:100]+"...")
print(f"⏱ Tempo prima parola: {t_fine-t_start:.2f}s (tutto insieme)\n")
print(risposta.content[0].text)
# ── CON streaming ─────────────────────────────────────────────────
print("CON streaming (con flush=True):")
t_start = time.time()
primo_token = True

# TODO: usate client.messages.stream() con un context manager
# Per ogni text in stream.text_stream:
#   - se è il primo token, stampate il tempo trascorso
#   - stampate il testo con end="" e flush=True
full_text = ""
with client.messages.stream(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    messages=[{"role": "user", "content": domanda_lunga}]
) as stream:
    for text in stream.text_stream:
        if primo_token:
            print(f"⏱ Primo token: {time.time()-t_start:.2f}s")
            primo_token = False
            print(text, end="", flush=True)
            full_text += text



print(risposta.content[0].text)
print("💡 Ora togliete flush=True e rieseguite. Cosa cambia?")

SENZA streaming:
# Sistema IoT Completo: Guida Dettagliata

## 1️⃣ LIVELLO DEI SENSORI (Edge)

### Raccolta dei dati
...
⏱ Tempo prima parola: 2.52s (tutto insieme)

# Sistema IoT Completo: Guida Dettagliata

## 1️⃣ LIVELLO DEI SENSORI (Edge)

### Raccolta dei dati
```
┌─────────────────────────────────────┐
│      SENSORI FISICI                 │
├─────────────────────────────────────┤
│ • Temperatura    → NTC, DHT22       │
│ • Umidità        → Capacitivi       │
│ • Movimento      → PIR              │
│ • Pressione      → BMP280           │
│ • CO2            → MH-Z19           │
└─────────────────────────────────────┘
           ↓
┌─────────────────────────────────────┐
│    MICROCONTROLLORE (Edge Device)   │
├─────────────────────────────────────┤
│ • Arduino, Raspberry Pi             │
│ • ESP32, STM32                      │
│ • Acquisizione dati (ADC)           │
│ • Elaborazione locale               │
│ • Filtaggio anomalie                │
└────────────────────────────────────

---
### Esercizio 3 — Chatbot completo: memoria + streaming *(libero)*

Combinate tutto: history persistente su JSON + streaming.
Il chatbot deve:
- Ricordare tra sessioni diverse
- Rispondere in streaming
- Avere un system prompt WiData
- Salvare dopo ogni messaggio

In [ ]:
import os
import json
import anthropic

client = anthropic.Anthropic()

MEMORY_FILE_V2 = "chat_widata_v2.json"

SYSTEM_WIDATA = """
Sei l'assistente virtuale di WiData Srl, startup IoT di Sassari.
Rispondi in modo professionale su prodotti e servizi IoT.
Ricorda le informazioni che l'utente ti fornisce durante la conversazione.
"""


def carica_storia():
    if os.path.exists(MEMORY_FILE_V2):
        with open(MEMORY_FILE_V2, "r", encoding="utf-8") as f:
            storia = json.load(f)
            print(f"📂 Storia caricata: {len(storia)} messaggi precedenti")
            return storia
    print("🆕 Nessuna storia precedente — nuova conversazione")
    return []


def salva_storia(history):
    with open(MEMORY_FILE_V2, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    print(f"💾 Storia salvata: {len(history)} messaggi")


def chat_completo(messaggio, history):
    history.append({"role": "user", "content": messaggio})

    print("🤖 ", end="", flush=True)
    testo_completo = ""

    with client.messages.stream(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        system=SYSTEM_WIDATA,
        messages=history,
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
            testo_completo += text

    history.append({"role": "assistant", "content": testo_completo})
    salva_storia(history)

    print()
    return testo_completo


# --- Sessione 1 ---
history = carica_storia()
chat_completo("Ciao! Sto valutando i vostri sensori per un progetto smart city.", history)
chat_completo("Ho un budget di €5000 per 50 punti di monitoraggio.", history);

# --- Sessione 2 (decommentare dopo aver riavviato il kernel) ---
history = carica_storia()
chat_completo("Ricordi il mio budget e quanti punti devo monitorare?", history);

🆕 Nessuna storia precedente — nuova conversazione
🤖 Ciao! Benvenuto! 👋

Sono entusiasta di aiutarti con la tua iniziativa smart city. WiData offre soluzioni IoT affidabili e scalabili proprio per applicazioni come la tua.

Per guidarti al meglio verso i sensori più adatti, sarebbe utile conoscere alcuni dettagli del tuo progetto:

1. **Ambito applicativo**: Quali aspetti della città vuoi monitorare?
   - Qualità dell'aria e ambientale?
   - Traffico e mobilità urbana?
   - Illuminazione pubblica?
   - Gestione dei rifiuti?
   - Altro?

2. **Scala del progetto**: Stai pensando a un'area pilota o a una copertura cittadina diffusa?

3. **Infrastruttura**: Hai già una piattaforma di gestione dati, o cerchi una soluzione completa end-to-end?

4. **Esigenze specifiche**: Quali parametri vuoi misurare esattamente?

Raccontami di più e ti proporrò le soluzione più idonee per il tuo caso! 🏙️💾 Storia salvata: 2 messaggi

🤖 Ottimo! 📊 Ho preso nota del tuo budget di **€5000 per 50 punti di monitor

---
### Esercizio 4 — Gestione errori e robustezza *(libero)*

Cosa succede se il file JSON è corrotto?
E se l'utente interrompe lo streaming a metà?

Rendete il chatbot robusto aggiungendo gestione degli errori.

In [ ]:
# Esercizio 4 — robustezza
import json
# Caso 1: file JSON corrotto
# Simulate un file corrotto e verificate che carica_storia() gestisca l'errore
with open("chat_corrotto.json", "w") as f:
    f.write("{questo non è json valido")

def carica_storia_robusta(filepath):
    """Versione robusta che gestisce file corrotti."""
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            storia = json.load(f)
            return storia
    except (json.JSONDecodeError) as e:
        print(f"⚠️ Errore durante il caricamento della storia: {e}")
        return []
    # TODO: aggiungete try/except per json.JSONDecodeError
    # Se il file è corrotto, loggare l'errore e restituire []
    ___

history_test = carica_storia_robusta("chat_corrotto.json")
print(f"Caricamento file corrotto: {len(history_test)} messaggi (atteso: 0)")

# Caso 2: streaming interrotto
# Cosa succede alla history se lo streaming viene interrotto?
# Come gestite il messaggio parziale?

def chat_streaming_robusto(messaggio, history):
    """Streaming con gestione dell'interruzione."""
    history.append({"role": "user", "content": messaggio})
    testo_parziale = ""
    try:
        with client.messages.stream(
            model="claude-haiku-4-5-20251001",
            max_tokens=300,
            messages=history
        ) as stream:
            for text in stream.text_stream:
                testo_parziale += text
                print(text, end="", flush=True)
        history.append({"role": "user", "content": testo_parziale})
    except KeyboardInterrupt:
        # TODO: se l'utente interrompe, salvate comunque il testo parziale
        print("\n⚠️ Streaming interrotto")
        history.append({"role": "user", "content": testo_parziale})
        return testo_parziale
    finally:
        print()

history = [];
history_test2 = chat_streaming_robusto("Di cosa si occupa WiData",history)
risposta1 = chat_streaming_robusto("Di cosa si tratta l'Ai",history)

# Conclusione:
# Quali errori avete trovato?
# Come li avete risolti?
# ...

⚠️ Errore durante il caricamento della storia: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)
Caricamento file corrotto: 0 messaggi (atteso: 0)
# WiData

WiData è un'azienda che si occupa di **consulenza e servizi nel campo dei dati e dell'analisi**. Le sue principali aree di attività includono:

## Servizi principali

- **Data Strategy**: sviluppo di strategie per la gestione e valorizzazione dei dati
- **Data Analytics**: analisi avanzate per supportare le decisioni aziendali
- **Business Intelligence**: creazione di sistemi di reportistica e dashboard
- **Data Governance**: organizzazione e gestione della qualità dei dati
- **Data Science**: implementazione di modelli predittivi e machine learning

## Clienti e settori

Lavora con aziende di varie dimensioni, supportandole nella:
- Trasformazione digitale
- Ottimizzazione dei processi
- Monetizzazione dei dati

---

**Nota**: Se stai cercando informazioni specifiche su una particolare divisione o progett

# 🏆 Esercizio Bonus — Chatbot completo in una cella

Avete finito prima? Bene. Questo esercizio integra **tutto quello che avete imparato oggi** in un unico chatbot funzionante.

---

## ✅ Requisiti da implementare

| # | Requisito |
|---|-----------|
| 1 | **Memoria persistente** su JSON — carica all'avvio, salva dopo ogni messaggio |
| 2 | **Sliding window** — manda solo gli ultimi `MAX_TURNI` turni al modello |
| 3 | **Streaming** con `flush=True` |
| 4 | **System prompt** WiData |
| 5 | **Contatore token** prima di ogni invio |
| 6 | Funzione **`reset()`** per cancellare la storia |
| 7 | Funzione **`storia()`** per vedere i messaggi salvati |

---

## 🧪 Test da superare

```
A. chat("Ciao, mi chiamo Sara.")
B. chat("Cosa fai?")
C. chat("Come mi chiamo?")       → deve rispondere "Sara"
D. Riavvia il kernel e riesegui la cella
E. chat("Ti ricordi di me?")     → deve ricordare Sara
F. reset(), poi storia()         → deve essere vuota
```

---

## 💡 Suggerimenti

- `ensure_ascii=False` è obbligatorio per i caratteri italiani
- `count_tokens()` vuole la **finestra già applicata**, non tutta la history
- `flush=True` è obbligatorio per lo streaming token per token
- `MAX_TURNI * 2` perché ogni turno = 1 messaggio `user` + 1 messaggio `assistant`

In [1]:
import json
import os

from google.colab import userdata
import anthropic


# ── Configurazione ────────────────────────────────────────────────
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

MEMORY_FILE = "chatbot_widata_bonus.json"
MAX_TURNI = 10
MODEL = "claude-haiku-4-5-20251001"

SYSTEM = """Sei l'assistente virtuale di WiData Srl, startup IoT di Sassari.
Rispondi in modo professionale su prodotti e servizi IoT.
Ricorda le informazioni fornite durante la conversazione."""



def carica_storia():
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def salva_storia(history):
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)


def storia(history):
  with open(MEMORY_FILE, "r", encoding="utf-8") as f:
    print(f)

def finestra(history):
    return history[-MAX_TURNI * 2:]



def chat(messaggio):
    history = carica_storia()


    messaggi_da_contare = finestra(history) + [{"role": "user", "content": messaggio}]
    token_pre = client.messages.count_tokens(
        model=MODEL,
        system=SYSTEM,
        messages=messaggi_da_contare,
    ).input_tokens
    print(f"[📊 {token_pre} token | {len(history)} messaggi in storia]\n")


    history.append({"role": "user", "content": messaggio})
    messaggi_da_inviare = finestra(history)


    print("🤖 ", end="", flush=True)
    testo = ""
    with client.messages.stream(
        model=MODEL,
        max_tokens=400,
        system=SYSTEM,
        messages=messaggi_da_inviare,
    ) as stream:
        for text in stream.text_stream:
            testo += text
            print(text, end="", flush=True)
    print("\n")

    history.append({"role": "assistant", "content": testo})
    salva_storia(history)
    return testo



def reset():
    if os.path.exists(MEMORY_FILE):
        os.remove(MEMORY_FILE)
    print("🗑️ Storia cancellata.")


def storia():
    h = carica_storia()
    if not h:
        print("Nessun messaggio.")
        return
    for i, m in enumerate(h, start=1):
        ruolo = "Tu" if m["role"] == "user" else "🤖"
        print(f"[{i}] {ruolo}: {m['content']}")



h = carica_storia()
print(f"🤖 Chatbot pronto. {len(h)} messaggi in storia.")
print("Usa: chat('messaggio') | storia() | reset()\n")

#storia()
#reset()
chat("Ti ricordi di me?")

🤖 Chatbot pronto. 8 messaggi in storia.
Usa: chat('messaggio') | storia() | reset()

[📊 926 token | 8 messaggi in storia]

🤖 Sì, Sara! 😊

Mi ricordo che **ti chiami Sara** e che stiamo conversando in questa sessione.

Finora abbiamo parlato di:
- Chi sono e cosa faccio come assistente di WiData
- La mia funzione di supporto su soluzioni IoT

Come ti ho spiegato, **ricordo le informazioni all'interno di questa conversazione**, ma non avrò memoria se tornerai a contattarmi in futuro in una nuova sessione.

**C'è qualcosa di nuovo che ti piacerebbe sapere su WiData o sulle nostre soluzioni IoT?** Sono qui per aiutarti! 🚀



"Sì, Sara! 😊\n\nMi ricordo che **ti chiami Sara** e che stiamo conversando in questa sessione.\n\nFinora abbiamo parlato di:\n- Chi sono e cosa faccio come assistente di WiData\n- La mia funzione di supporto su soluzioni IoT\n\nCome ti ho spiegato, **ricordo le informazioni all'interno di questa conversazione**, ma non avrò memoria se tornerai a contattarmi in futuro in una nuova sessione.\n\n**C'è qualcosa di nuovo che ti piacerebbe sapere su WiData o sulle nostre soluzioni IoT?** Sono qui per aiutarti! 🚀"

In [19]:
import json, os
from google.colab import userdata
import anthropic

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()

MEMORY_FILE = "chatbot_widata_bonus.json"
MAX_TURNI = 10
SYSTEM = """
Sei l'assistente virtuale di WiData Srl, startup IoT di Sassari.
Rispondi in modo professionale su prodotti e servizi IoT.
Ricorda le informazioni fornite durante la conversazione.
"""
history = []
def carica_storia():
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

def salva_storia(history):
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

def chat(messaggio):
    history = carica_storia()
    finestra = history[-MAX_TURNI*2:]
    token_pre = client.messages.count_tokens(
        model="claude-haiku-4-5-20251001",
        messages=finestra + [{"role":"user","content":messaggio}]
    ).input_tokens if finestra or messaggio else 0
    print(f"[📊 {token_pre} token | {len(history)} messaggi in storia]\n")
    history.append({"role": "user", "content": messaggio})
    finestra = history[-MAX_TURNI*2:]
    print("🤖 ", end="", flush=True)
    testo = ""
    with client.messages.stream(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        system=SYSTEM,
        messages=finestra
    ) as stream:
        for text in stream.text_stream:
            testo += text
            print(text, end="", flush=True)
    print("\n")
    history.append({"role": "assistant", "content": testo})
    salva_storia(history)

def chat_completo(messaggio, history):
    history.append({"role": "user", "content": messaggio})

    print("🤖 ", end="", flush=True)
    testo_completo = ""

    with client.messages.stream(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        system=SYSTEM_WIDATA,
        messages=history,
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
            testo_completo += text

    history.append({"role": "assistant", "content": testo_completo})
    salva_storia(history)

    print()
    return testo_completo


def reset():
    if os.path.exists(MEMORY_FILE): os.remove(MEMORY_FILE)
    print("🗑️  Storia cancellata.")

def storia():
    h = carica_storia()
    if not h: print("Nessun messaggio."); return
    for i, m in enumerate(h):
        ruolo = "Tu" if m["role"] == "user" else "🤖"
        print(f"[{i+1}] {ruolo}: {m['content'][:80]}...")

# ── Avvio ──────────────────────────────────────────────────────────
history = carica_storia()
print(f"🤖 Chatbot pronto. {len(history)} messaggi in storia.")
print("Usa: chat('messaggio') | storia() | reset()\n")

# ── Modifica qui il messaggio e riesegui la cella ──────────────────
chat_completo("Ciao mi chiamo Sara")


🤖 Chatbot pronto. 14 messaggi in storia.
Usa: chat('messaggio') | storia() | reset()



TypeError: chat_completo() missing 1 required positional argument: 'history'

---
## 📊 Preparate la presentazione (5 slide)

1. **Memoria persistente** — come funziona, demo chiudi-riapri-ricorda
2. **ensure_ascii=False** — mostrate cosa succede senza (con accenti italiani)
3. **Streaming** — il confronto tempo primo token con e senza
4. **flush=True** — mostrate cosa succede senza
5. **Il chatbot completo** — demo live del chatbot con memoria + streaming

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*